In [8]:
#! mlflow server -p 5001

In [9]:
import mlflow
from openai import OpenAI
from loguru import logger

In [10]:
PORT = "5001"
EXPERIMENT_NAME = "Chatbot"
TRACE_ID = "id-12345"
TRACE_KEY = "chatbot_version"
TRACE_VALUE = "1.0"

In [11]:
# Set up MLflow autologging and tracking URI
mlflow.openai.autolog()
mlflow.set_tracking_uri(f"http://localhost:{PORT}")
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1774008395965, experiment_id='3', last_update_time=1774008395965, lifecycle_stage='active', name='Chatbot', tags={}, workspace='default'>

In [12]:
OLLAMA_API_URL = "http://localhost:11434/v1"
LLM_MODEL = "gpt-oss:20b"
TEMPERATURE = 0.1
MAX_TOKENS = 100

In [13]:
class Agent:
    def __init__(self, system=""):
        self.client = self._set_llm(OLLAMA_API_URL)
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})
            logger.info(f"System message set: {system}")

    def _set_llm(self, url):
        try:
            logger.info(f"Ollama API URL: {url}")
            return OpenAI(base_url=url, 
                          api_key="dummy")
        except Exception as e:
            print(f"Error initializing OpenAI client: {e}")
            return None

    def __call__(self, message):
        try:
            self.messages.append({"role": "user", "content": message})
            logger.info(f"User message added: {message}")
            result = self.execute()
            self.messages.append({"role": "assistant", "content": result})
            logger.info(f"Assistant response added: {result}")
            return result
        except Exception as e:
            logger.error(f"Error during agent call: {e}")
            return "An error occurred while processing your request."

    def execute(self):
        try:
            response = self.client.chat.completions.create(
                        model=LLM_MODEL,
                        messages=self.messages,
                        temperature=TEMPERATURE,
                        max_tokens=MAX_TOKENS,
                    )
            return response.choices[0].message.content
        except Exception as e:
            logger.error(f"Error during agent execution: {e}")
            return "An error occurred while processing your request."

## AI Chatbot

In [ ]:
from mlflow.entities import SpanType

@mlflow.trace(span_type=SpanType.CHAIN)
def start_session():
    agent = Agent(system="You are a helpful assistant.")
    while True:
        with mlflow.start_span(name="User") as span:
            user_input = input("You: ")
        if user_input == "BYE":
            break
        with mlflow.start_span(name="Agent") as span:
            response = agent(user_input)
            print(f"Agent: {response}")

start_session()

2026-03-20 13:29:54.569 | INFO     | __main__:_set_llm:12 - Ollama API URL: http://localhost:11434/v1
2026-03-20 13:29:54.575 | INFO     | __main__:__init__:8 - System message set: You are a helpful assistant.
2026-03-20 13:29:59.085 | INFO     | __main__:__call__:22 - User message added: ciao, come stai?
2026-03-20 13:30:14.704 | INFO     | __main__:__call__:25 - Assistant response added: Ciao! Sto bene, grazie. E tu?


Agent: Ciao! Sto bene, grazie. E tu?


2026-03-20 13:30:41.113 | INFO     | __main__:__call__:22 - User message added: bene, grazie. mi sai dire qual'è il mezzo di trasporto più ecologico?
2026-03-20 13:30:46.146 | INFO     | __main__:__call__:25 - Assistant response added: Ciao!  
Il mezzo di trasporto più ecologico dipende dal contesto, ma in generale **camminare e andare in bicicletta** sono


Agent: Ciao!  
Il mezzo di trasporto più ecologico dipende dal contesto, ma in generale **camminare e andare in bicicletta** sono


2026-03-20 13:31:02.371 | INFO     | __main__:__call__:22 - User message added: sono?
2026-03-20 13:31:07.443 | INFO     | __main__:__call__:25 - Assistant response added: **Camminare** e **andare in bicicletta** sono i mezzi di trasporto più ecologici perché:

| Mezzo | Emissioni di CO₂ (per km) | Vantaggi | Svantaggi |
|-------|--------------------------|----------|-----------|
| **Camminare** |


Agent: **Camminare** e **andare in bicicletta** sono i mezzi di trasporto più ecologici perché:

| Mezzo | Emissioni di CO₂ (per km) | Vantaggi | Svantaggi |
|-------|--------------------------|----------|-----------|
| **Camminare** |


Trace(trace_id=tr-6429d6d9b070041849355f2c1a847bfb)